# Sleeper Marts Pipeline

Business-friendly data marts for reporting and analysis.

These views aggregate and simplify data from core and trades schemas for easy consumption.

In [ ]:
-- =============================================================================
-- SLEEPER MARTS PIPELINE
-- Business-Friendly Data Marts
-- 
-- Purpose: Simplified views for reporting and analysis
-- These marts aggregate data from core and trades schemas
-- =============================================================================


In [ ]:
-- ---------- SEASON OVERVIEW ----------


In [ ]:
-- Season standings with consistency metrics
CREATE OR REPLACE MATERIALIZED VIEW mart_season_overview AS
WITH last_week AS (
  SELECT league_id, season, roster_id, max(week) AS max_week
  FROM workspace.sleeper_core.fact_standings_week
  GROUP BY league_id, season, roster_id
),
asof AS (
  SELECT s.*
  FROM workspace.sleeper_core.fact_standings_week s
  JOIN last_week m
    ON s.league_id=m.league_id 
    AND s.season=m.season 
    AND s.roster_id=m.roster_id 
    AND s.week=m.max_week
),
league_info AS (
  SELECT 
    league_id,
    season,
    name AS league_name,
    lower(regexp_replace(coalesce(name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
)
SELECT 
  li.cluster_key,
  li.league_name,
  a.league_id, 
  a.season, 
  a.roster_id,
  mr.manager_display_name,
  a.wins, 
  a.losses, 
  a.ties,
  a.points_for, 
  a.points_against, 
  a.expected_wins,
  cm.points_for_stddev, 
  cm.points_for_cv, 
  cm.games_played
FROM asof a
LEFT JOIN workspace.sleeper_core.agg_consistency_metrics cm
  ON a.league_id=cm.league_id 
  AND a.season=cm.season 
  AND a.roster_id=cm.roster_id
LEFT JOIN league_info li 
  ON a.league_id = li.league_id 
  AND a.season = li.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
  ON a.league_id = mr.league_id
  AND a.season = mr.season
  AND a.roster_id = mr.roster_id;

In [ ]:
-- ---------- TRADE ANALYSIS ----------


In [ ]:
-- Trade winners leaderboard (all-time career value)
CREATE OR REPLACE MATERIALIZED VIEW mart_trade_winners AS
SELECT
  cluster_key,
  cluster_name,
  league_id,
  transaction_id,
  trade_season,
  roster_a,
  roster_b,
  roster_a_manager,
  roster_b_manager,
  roster_a_points,
  roster_b_points,
  point_differential,
  winner_manager,
  loser_manager,
  trade_impact_magnitude,
  trade_is_complete
FROM workspace.sleeper_trades.agg_trade_winners_enriched
WHERE trade_is_complete = TRUE
ORDER BY trade_impact_magnitude DESC;

In [ ]:
-- Trade impact by manager and season
CREATE OR REPLACE MATERIALIZED VIEW mart_trade_impact_by_manager AS
WITH trade_summary AS (
  SELECT
    cluster_key,
    cluster_name,
    league_id,
    trade_season,
    side_roster_id,
    -- Aggregate across all trades
    COUNT(DISTINCT transaction_id) AS total_trades,
    SUM(same_season_points) AS total_same_season_points,
    SUM(year1_points) AS total_year1_points,
    SUM(career_points) AS total_career_points
  FROM workspace.sleeper_trades.agg_trade_impact_summary
  WHERE trade_is_complete = TRUE
  GROUP BY cluster_key, cluster_name, league_id, trade_season, side_roster_id
)
SELECT
  ts.*,
  mr.manager_display_name
FROM trade_summary ts
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
  ON ts.league_id = mr.league_id
  AND ts.trade_season = mr.season
  AND ts.side_roster_id = mr.roster_id
ORDER BY ts.cluster_key, ts.trade_season, ts.total_career_points DESC;

In [ ]:
-- ---------- PLAYER PERFORMANCE ----------


In [ ]:
-- Player performance with names and league context
CREATE OR REPLACE MATERIALIZED VIEW mart_player_performance AS
WITH league_info AS (
  SELECT 
    league_id,
    season,
    name AS league_name,
    lower(regexp_replace(coalesce(name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
)
SELECT 
  li.cluster_key,
  li.league_name,
  f.league_id, 
  f.season, 
  f.week, 
  f.roster_id,
  mr.manager_display_name,
  f.player_id,
  p.full_name AS player_name,
  p.position, 
  f.points,
  f.was_started,
  f.projected_points
FROM workspace.sleeper_core.fact_player_week f
LEFT JOIN workspace.sleeper_core.dim_players p
  ON f.player_id = p.player_id
LEFT JOIN league_info li
  ON f.league_id = li.league_id
  AND f.season = li.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
  ON f.league_id = mr.league_id
  AND f.season = mr.season
  AND f.roster_id = mr.roster_id;

In [ ]:
-- Player career totals per roster
CREATE OR REPLACE MATERIALIZED VIEW mart_player_roster_totals AS
WITH league_info AS (
  SELECT 
    league_id,
    lower(regexp_replace(coalesce(name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
    name AS league_name
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
),
-- Get the most common league name for each cluster
cluster_names AS (
  SELECT 
    cluster_key,
    any_value(league_name) AS cluster_name
  FROM league_info
  GROUP BY cluster_key
)
SELECT
  cn.cluster_key,
  cn.cluster_name,
  rt.league_id,
  rt.roster_id,
  rt.player_id,
  rt.full_name AS player_name,
  rt.position,
  rt.first_season,
  rt.last_season,
  rt.seasons_count,
  rt.weeks_count,
  rt.total_points,
  rt.points_per_week,
  rt.points_as_starter
FROM workspace.sleeper_core.agg_player_roster_totals rt
JOIN league_info li ON rt.league_id = li.league_id
JOIN cluster_names cn ON li.cluster_key = cn.cluster_key;

In [ ]:
-- ---------- WAIVER WIRE ACTIVITY ----------


In [ ]:
-- Waiver acquisitions with player and manager details
CREATE OR REPLACE MATERIALIZED VIEW mart_waiver_activity AS
WITH league_info AS (
  SELECT 
    league_id,
    season,
    name AS league_name,
    lower(regexp_replace(coalesce(name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
)
SELECT
  li.cluster_key,
  li.league_name,
  w.league_id,
  w.season,
  w.week,
  w.transaction_id,
  w.roster_id,
  mr.manager_display_name,
  w.player_id,
  p.full_name AS player_name,
  p.position,
  w.faab_bid,
  w.was_waiver,
  w.acquired_date
FROM workspace.sleeper_core.fact_waiver_acquisitions w
LEFT JOIN workspace.sleeper_core.dim_players p
  ON w.player_id = p.player_id
LEFT JOIN league_info li
  ON w.league_id = li.league_id
  AND w.season = li.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
  ON w.league_id = mr.league_id
  AND w.season = mr.season
  AND w.roster_id = mr.roster_id;

In [ ]:
-- ---------- DRAFT ANALYSIS ----------


In [ ]:
-- Draft ROI by round
CREATE OR REPLACE MATERIALIZED VIEW mart_draft_roi AS
WITH league_info AS (
  SELECT 
    league_id,
    season,
    name AS league_name,
    lower(regexp_replace(coalesce(name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
)
SELECT
  li.cluster_key,
  li.league_name,
  r.league_id,
  r.season,
  r.round,
  r.picks_count,
  r.avg_rookie_points,
  r.avg_career_points,
  r.season_avg_points,
  r.career_avg_points,
  r.rookie_roi,
  r.career_roi
FROM workspace.sleeper_core.agg_draft_roi_by_round r
LEFT JOIN league_info li
  ON r.league_id = li.league_id
  AND r.season = li.season;